# Fine-tune SmolVLA on the SO-101 pencil pick-and-place dataset

Run this on **Kaggle** with a **GPU** (Settings → Accelerator → GPU T4 x2 or P100) and **Internet ON**.

Before running, add your Hugging Face token as a Kaggle secret:
**Add-ons → Secrets → name it `HF_TOKEN`** (create the token at https://huggingface.co/settings/tokens, "write" scope).


## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install LeRobot (SmolVLA extra)
Installed from git to match the dataset format you recorded with.

In [ ]:
!pip install -q "lerobot[smolvla,dataset] @ git+https://github.com/huggingface/lerobot.git"

## 3. Log in to the Hugging Face Hub (via the `HF_TOKEN` Kaggle secret)

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

## 4. Settings — edit these

In [ ]:
HF_USER      = "your-hf-username"                       # <- your HF username
DATASET_REPO = f"{HF_USER}/so101_pencil_pickup"         # <- the dataset you recorded
MODEL_REPO   = f"{HF_USER}/smolvla_pencil_pickup"       # <- where to save the fine-tuned model

# Training hyper-parameters. T4 (16 GB): keep batch_size small (8-16) to avoid OOM.
BATCH_SIZE = 16
STEPS      = 20000      # ~20k is a good target; start with 10000 to validate the pipeline
SAVE_FREQ  = 5000       # checkpoint frequency (lets you resume across Kaggle's 12h session limit)
OUTPUT_DIR = "/kaggle/working/smolvla_pencil"

## 5. Fine-tune

Starts from the pretrained `lerobot/smolvla_base` (450M) and fine-tunes on your data.
On a T4 this is **slow** — 20k steps can exceed Kaggle's 12h limit, so checkpoints are saved every
`SAVE_FREQ` steps. If you run out of time, re-run and add `--resume=true` (and point `--config_path`
at the saved checkpoint) to continue.

In [ ]:
import subprocess
subprocess.run([
    "lerobot-train",
    "--policy.path=lerobot/smolvla_base",
    f"--dataset.repo_id={DATASET_REPO}",
    "--dataset.video_backend=pyav",
    f"--batch_size={BATCH_SIZE}",
    f"--steps={STEPS}",
    f"--save_freq={SAVE_FREQ}",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=smolvla_pencil",
    "--policy.device=cuda",
    "--policy.push_to_hub=false",
    "--wandb.enable=false",
], check=True)

## 6. Push the fine-tuned model to the Hub
So you can load it locally with `--policy.path=MODEL_REPO`.

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

ckpt = Path(OUTPUT_DIR) / "checkpoints" / "last" / "pretrained_model"
assert ckpt.exists(), f"checkpoint not found at {ckpt} - check the training output"

api = HfApi()
api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=str(ckpt), repo_id=MODEL_REPO, repo_type="model")
print("Pushed to https://huggingface.co/" + MODEL_REPO)

## Done
Back on your Mac, run the policy on the real arm:

```bash
cd pencil_pickup_vla && ./3_run_autonomous.sh
```
